In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langfuse import Langfuse

LANGFUSE_MAX_PAGE_SIZE = 100
DEFAULT_TRACE_LIMIT = 1000
DEFAULT_ENV_PATH = "/Users/daniel.hochmuth/code/mandarin_learner/.env"


def load_client() -> Langfuse:
    load_dotenv(dotenv_path=DEFAULT_ENV_PATH)

    base_url = os.getenv("LANGFUSE_HOST") or os.getenv("LANGFUSE_BASE_URL")
    public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
    secret_key = os.getenv("LANGFUSE_SECRET_KEY")

    if not public_key or not secret_key:
        raise ValueError("Missing LANGFUSE_PUBLIC_KEY or LANGFUSE_SECRET_KEY")

    if not base_url:
        raise ValueError("Missing LANGFUSE_HOST or LANGFUSE_BASE_URL")

    return Langfuse(
        public_key=public_key,
        secret_key=secret_key,
        base_url=base_url,
        host=base_url,
        tracing_enabled=False,
    )


def list_traces(client: Langfuse, limit: int = DEFAULT_TRACE_LIMIT):
    remaining = limit
    current_page = 1
    all_traces = []

    while remaining > 0:
        page_limit = min(remaining, LANGFUSE_MAX_PAGE_SIZE)
        response = client.api.trace.list(
            page=current_page,
            limit=page_limit,
            order_by="timestamp.desc",
        )

        batch = response.data
        all_traces.extend(batch)
        remaining -= len(batch)

        if len(batch) < page_limit:
            break

        current_page += 1

    return all_traces


In [2]:
client = load_client()
traces = list_traces(client)

In [ ]:
traces_df = pd.DataFrame([trace.model_dump(by_alias=True) for trace in traces])

# Convert timestamp columns
for column in ("timestamp", "createdAt", "updatedAt"):
    if column in traces_df.columns:
        traces_df[column] = pd.to_datetime(traces_df[column], errors="coerce")

print(f"Loaded {len(traces_df)} traces")
traces_df.head()

Loaded 14 traces


,id,timestamp,name,input,output,session_id,release,version,user_id,metadata,...,html_path,latency,total_cost,observations,scores,projectId,bookmarked,createdAt,updatedAt,externalId
0,000e26caee3a0e89178734639ff1a443,2026-03-22 15:52:32.025000+00:00,generate_sentences,"{'args': [], 'kwargs': {'characters_required':...","{'valid': True, 'attempts': 1, 'sentences': [{...",None,None,None,None,{'resourceAttributes': {'telemetry.sdk.languag...,...,/project/cmn0qyry500fqad07qw9qucb2/traces/000e...,38.060,0.015836,"[ae47e737d2ccc943, 20b1b48cc8b7719c]",[],cmn0qyry500fqad07qw9qucb2,False,2026-03-22 15:53:14+00:00,2026-03-22 15:53:13.374000+00:00,None
1,25705e42e78d38f37d3a460058583339,2026-03-22 15:49:31.986000+00:00,generate_sentences,"{'args': [], 'kwargs': {'characters_required':...","{'valid': True, 'attempts': 1, 'sentences': [{...",None,None,None,None,{'resourceAttributes': {'telemetry.sdk.languag...,...,/project/cmn0qyry500fqad07qw9qucb2/traces/2570...,14.152,0.012709,"[885d607502fbd348, 3c9a115b8b07ba91]",[],cmn0qyry500fqad07qw9qucb2,False,2026-03-22 15:49:48+00:00,2026-03-22 15:49:47.954000+00:00,None
2,073885f4583b1afcf2da55ed5744e3fd,2026-03-22 15:45:57+00:00,generate_sentences,"{'args': [], 'kwargs': {'characters_required':...","{'valid': True, 'attempts': 1, 'sentences': [{...",None,None,None,None,{'resourceAttributes': {'telemetry.sdk.languag...,...,/project/cmn0qyry500fqad07qw9qucb2/traces/0738...,12.849,0.009769,"[1c30f99a978feb9b, 81114337ed7c8a96]",[],cmn0qyry500fqad07qw9qucb2,False,2026-03-22 15:46:13+00:00,2026-03-22 15:46:12.456000+00:00,None
3,6534969dd42184bc65df0d359562f76f,2026-03-22 15:38:14.598000+00:00,generate_sentences,"{'args': [], 'kwargs': {'characters_required':...","{'valid': True, 'attempts': 1, 'sentences': [{...",None,None,None,None,{'resourceAttributes': {'telemetry.sdk.languag...,...,/project/cmn0qyry500fqad07qw9qucb2/traces/6534...,25.871,0.019623,"[308843fc534275ea, b57ed932ac1c3248]",[],cmn0qyry500fqad07qw9qucb2,False,2026-03-22 15:38:44+00:00,2026-03-22 15:38:43.758000+00:00,None
4,83fc8b8f8605f44aaf8943361d2a9a71,2026-03-22 15:37:23.691000+00:00,generate_sentences,"{'args': [], 'kwargs': {'characters_required':...","{'valid': True, 'attempts': 1, 'sentences': [{...",None,None,None,None,{'resourceAttributes': {'telemetry.sdk.languag...,...,/project/cmn0qyry500fqad07qw9qucb2/traces/83fc...,16.382,0.012711,"[16ceb52fff99627f, 50869565b3b447ed]",[],cmn0qyry500fqad07qw9qucb2,False,2026-03-22 15:37:43+00:00,2026-03-22 15:37:42.976000+00:00,None


In [28]:
def extract_input_col(row, name):
    try:
        return row["kwargs"][name]
    except:
        return None

In [44]:
def check_input_validity(row):
    return (
        isinstance(row["characters_required"], list) and len(row["characters_required"]) > 0
        and isinstance(row["characters_optional"], list) and len(row["characters_optional"]) > 0
    )

In [45]:
def check_output_validity(row):
    return (
        isinstance(row["sentence_chinese"], str) and len(row["sentence_chinese"]) > 0
    )

In [69]:
import re

def extract_chinese(text):
    if not isinstance(text, str):
        return ""
    return "".join(re.findall(r'[\u4e00-\u9fff]', text))

In [71]:
def contains_all_required(row):
    required = row["characters_required"]
    sentence = row["sentence_chinese"]
    
    return (
        isinstance(required, list)
        and isinstance(sentence, str)
        and all(char in sentence for char in required)
    )

In [73]:
def sentence_uses_allowed_chars(row):
    sentence = row["sentence_chinese_hanzi_only"]
    required = row["characters_required"]
    optional = row["characters_optional"]

    if not isinstance(sentence, str):
        return False
    if not isinstance(required, list) or not isinstance(optional, list):
        return False

    allowed_chars = set(required).union(set(optional))

    return all(char in allowed_chars for char in sentence)

In [77]:
sentences_df = traces_df.copy()
sentences_df["sentences"] = sentences_df["output"].apply(
    lambda output: output.get("sentences", []) if isinstance(output, dict) else []
)

sentences_df = sentences_df.explode("sentences")
sentences_df = sentences_df.reset_index()

sentences_df["characters_required"] = sentences_df["input"].apply(lambda x: extract_input_col(x, "characters_required"))
sentences_df["characters_optional"] = sentences_df["input"].apply(lambda x: extract_input_col(x, "characters_optional"))
sentences_df["sentence_chinese"] = sentences_df["sentences"].apply(lambda x: x["chinese"])
sentences_df["sentence_chinese_hanzi_only"] = sentences_df["sentence_chinese"].apply(extract_chinese)

sentences_df["valid_input"] = sentences_df.apply(check_input_validity, axis=1)
sentences_df["valid_output"] = sentences_df.apply(check_output_validity, axis=1)

sentences_df["all_present"] = sentences_df.apply(contains_all_required, axis=1)
sentences_df["only_allowed"] = sentences_df.apply(sentence_uses_allowed_chars, axis=1)

In [78]:
sentences_df["all_present"].sum()

np.int64(8)

In [79]:
sentences_df["only_allowed"].sum()

np.int64(30)

In [75]:
sentences_df[["all_present", "only_allowed", "characters_required", "sentence_chinese"]]

,all_present,only_allowed,characters_required,sentence_chinese
0,True,True,[脑],大脑会想。
1,True,True,[脑],他的大脑很好。
2,False,True,[脑],我看书。
3,False,True,[脑],你听吗？
4,False,True,[脑],我们在家写。
5,False,True,"[呢, 习]",我学习。
6,False,True,"[呢, 习]",你呢？
7,False,True,"[有, 这]",这是我家。
8,False,True,"[有, 这]",有水。
9,False,True,"[有, 这, 那, 哪, 饭, 没, 呢, 习]",哪有饭？
